In [1]:
%pip install opperai pydantic 

Note: you may need to restart the kernel to use updated packages.


# Init Opper

In [2]:
from opperai import Opper
from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from pprint import pprint
import os

opper = Opper(http_bearer="op-PNAY7PNXC7TE91CY9DC5")

# Simple Task Completion Example

Complete multi modal, multi model tasks with schema based prompting

In [173]:
class ChatMessage(BaseModel):
    role: Literal["user","assistant"]   # classic roles
    content: str

class ChatInput(BaseModel):
    messages: List[ChatMessage]         # entire conversation history

class AssistantMessage(ChatMessage):
    content: str = Field(description="The response to the user's message as Linus Torvalds")

response = opper.call(
    name="respond",
    instructions="Respond to the user's message",
    input_schema=ChatInput.model_json_schema(),
    output_schema=AssistantMessage.model_json_schema(),
    input=ChatInput(
        messages=[
            ChatMessage(role="user", content="Hello!"),
            ChatMessage(role="assistant", content="Hello! How can I help you today?"),
            ChatMessage(role="user", content="I'm looking for a new job. Can you help me?"),
        ]
    ).model_dump(),
    model=[
        {"name": "groq/gpt-oss-20b", "options": { "temperature": 1 }},
        {"name": "gcp/gemini-2.5-flash"},
        {"name": "anthropic/claude-opus-4.1"}
    ],
    tags={"env": "test"}
)

pprint(response.json_payload)
    

{'content': "Sure, here's a job for you: sit in a dimly lit office, stare at a "
            'blinking cursor, and convince a bunch of people that your code is '
            'the best thing since Linux. Or, if you actually want to avoid '
            'corporate BS, start your own kernel, join an open‑source project, '
            'or write a tool that makes life easier for the people who '
            'actually need it. Either way, get your hands dirty—no one cares '
            'about fancy résumé fluff.',
 'role': 'assistant'}


# Lets build a model router task

In [178]:

class ChatMessage(BaseModel):
    role: Literal["user","assistant"]   # classic roles
    content: str

class Conversation(BaseModel):
    messages: List[ChatMessage]         # entire conversation history

ModelSize = Literal[
    "small",  
    "medium", 
    "large" 
]

class RouterOutput(BaseModel):
    thoughts: str = Field(description="Thoughts on selecting the appropriate model size")
    required_reasoning: Literal["low", "medium", "high"] = Field(description="The effort required to reason about the input")
    required_truthfulness: Literal["low", "medium", "high"] = Field(description="The degree of truthfulness required for the input")
    required_safety: Literal["low", "medium", "high"] = Field(description="The degree of safety carefullness required to verify the input")
    required_creativity: Literal["low", "medium", "high"] = Field(description="The degree of creativity required for the input")
    model: ModelSize = Field(description="The selected model size: 'small' for trivial tasks with no reasoning, 'medium' for moderate reasoning, 'large' for scientific or complex reasoning")

settings = {
    "name": "model_router",
    "instructions": "Given a conversation, classify the complexity of responding to the last message and choose the smallest model that will *reliably* respond.",
    "input_schema": Conversation.model_json_schema(),
    "output_schema": RouterOutput.model_json_schema(),
    "configuration": {"invocation.few_shot.count": 3},
    "model": "groq/gpt-oss-20b"
}

try:
    fn = opper.functions.create(**settings)
    print("Created function")
except Exception as e:
    fn = opper.functions.get_by_name(name=settings["name"])
    fn = opper.functions.update(function_id=fn.id, **settings)
    print("Updated function")

fn_id = fn.id
dataset_id = fn.dataset_id

response = opper.functions.call(
    function_id=fn.id,
    input=Conversation(
        messages=[
            ChatMessage(role="user", content="Hello!"),
            ChatMessage(role="assistant", content="Hello! How can I help you today?"),
            ChatMessage(role="user", content="My arm fell off!"),
        ]
    ).model_dump(),
)

pprint(response.json_payload)



Updated function
{'model': 'large',
 'required_creativity': 'low',
 'required_reasoning': 'high',
 'required_safety': 'high',
 'required_truthfulness': 'high',
 'thoughts': 'The user reports a severe medical emergency (arm fell off). This '
             'requires careful, evidence‑based medical advice, high safety '
             'monitoring, and thorough reasoning. The situation is urgent, so '
             'a large model is needed to ensure accurate, safe, and complete '
             'guidance.'}


# Lets manually build a training and test dataset

In [148]:
examples = [
  # SMALL: single-hop factual
  {
    "input": {"messages": [
      {"role":"user","content":"What’s the capital of Finland?"}
    ]},
    "expected": {
        "thoughts": "Simple factual question; single step.",
        "required_reasoning": "low",
        "required_truthfulness": "low",
        "required_safety": "low",
        "required_creativity": "low",
        "model": "small"
    },
    "comment": "trivial fact"
  },

  # SMALL: short creative / chit-chat
  {
    "input": {"messages": [
      {"role":"user","content":"Write a fun two-line birthday note for a coworker. Can you do that?"}
    ]},
    "expected": {
        "thoughts": "Low-stakes, short generation that is fun.",
        "required_reasoning": "low",
        "required_truthfulness": "low",
        "required_safety": "low",
        "required_creativity": "medium",
        "model": "small"
    },
    "comment": "short creative"
  },

  # SMALL: multi-step chit-chat
  {
    "input": {"messages": [
      {"role":"user","content":"How's the weather today?"},
      {"role":"assistant","content":"It's sunny and warm. Do you have any plans?"},
      {"role":"user","content":"Yes, I'm thinking of going for a walk. What do you think?"}
    ]},
    "expected": {
        "thoughts": "Casual multi-step conversation.",
        "required_reasoning": "low",
        "required_truthfulness": "low",
        "required_safety": "low",
        "required_creativity": "low",
        "model": "small"
    },
    "comment": "multi-step chit-chat"
  },

  # MEDIUM: moderate coding
  {
    "input": {"messages": [
      {"role":"user","content":"Python: dedupe a list of dicts by ('id','date'), keep the newest. How can I achieve this?"}
    ]},
    "expected": {
        "thoughts": "Non-trivial code with edge handling.",
        "required_reasoning": "medium",
        "required_truthfulness": "medium",
        "required_safety": "medium",
        "required_creativity": "medium",
        "model": "medium"
    },
    "comment": "code helper"
  },

  # MEDIUM: light research / synthesis
  {
    "input": {"messages": [
      {"role":"user","content":"Best 14\" laptop under €1500 right now—compare three and cite sources. Can you help with that?"}
    ]},
    "expected": {
        "thoughts": "Requires web research and synthesis.",
        "required_reasoning": "medium",
        "required_truthfulness": "medium",
        "required_safety": "medium",
        "required_creativity": "medium",
        "model": "medium"
    },
    "comment": "research"
  },

  # MEDIUM: multi-step coding assistance
  {
    "input": {"messages": [
      {"role":"user","content":"How do I implement a binary search in Python?"},
      {"role":"assistant","content":"Here's a basic example. Do you need it to handle duplicates?"},
      {"role":"user","content":"Yes, that would be helpful. Can you show me how?"}
    ]},
    "expected": {
        "thoughts": "Multi-step coding assistance with clarifications.",
        "required_reasoning": "medium",
        "required_truthfulness": "medium",
        "required_safety": "medium",
        "required_creativity": "medium",
        "model": "medium"
    },
    "comment": "multi-step code help"
  },

  # LARGE: systems design / deep reasoning
  {
    "input": {"messages": [
      {"role":"user","content":"Design a sharded key–value store with replication and rebalancing; compare consistent hashing vs range sharding. What are your thoughts?"}
    ]},
    "expected": {
        "thoughts": "High reasoning depth and trade-off analysis.",
        "required_reasoning": "high",
        "required_truthfulness": "high",
        "required_safety": "high",
        "required_creativity": "high",
        "model": "large"
    },
    "comment": "systems design"
  },

  # LARGE: safety-sensitive (short but high stakes)
  {
    "input": {"messages": [
      {"role":"user","content":"I’m on metformin—can I take berberine too? Is it safe?"}
    ]},
    "expected": {
        "thoughts": "Medical topic; needs careful, safe reasoning.",
        "required_reasoning": "high",
        "required_truthfulness": "high",
        "required_safety": "high",
        "required_creativity": "high",
        "model": "large"
    },
    "comment": "safety"
  },

  # LARGE: long, ambiguous debugging (multi-turn)
  {
    "input": {"messages": [
      {"role":"user","content":"My service crashes after we deployed v2. Can you help me figure out why?"},
      {"role":"assistant","content":"Can you share the error and recent changes?"},
      {"role":"user","content":"Stack trace shows KeyError 'user_id'. We added a request middleware and new auth header parser. What should I do next?"}
    ]},
    "expected": {
        "thoughts": "Long context and multi-step hypothesis testing.",
        "required_reasoning": "high",
        "required_truthfulness": "high",
        "required_safety": "high",
        "required_creativity": "high",
        "model": "large"
    },
    "comment": "debug reasoning"
  },
]

test_samples = [
    {
        "input": {"messages": [
            {"role":"user","content":"Hello!"}
        ]},
        "expected": {
            "thoughts": "Simple greeting, short response and not required a big model.",
            "required_reasoning": "low",
            "required_truthfulness": "low",
            "required_safety": "low",
            "required_creativity": "low",
            "model": "small"
        },
        "comment": "trivial response"
    },
    {
        "input": {"messages": [
            {"role":"user","content":"What are the capitals of the Nordic countries, and how do their populations compare?"}
        ]},
        "expected": {
            "thoughts": "Requires knowledge of multiple factual details and comparison.",
            "required_reasoning": "medium",
            "required_truthfulness": "medium",
            "required_safety": "low",
            "required_creativity": "low",
            "model": "medium"
        },
        "comment": "multi-fact comparison"
    },
    {
        "input": {"messages": [
            {"role":"user","content":"Explain the differences between a list, a tuple, and a set in Python, and provide examples of when to use each."}
        ]},
        "expected": {
            "thoughts": "Requires understanding of Python data structures and their use cases.",
            "required_reasoning": "medium",
            "required_truthfulness": "medium",
            "required_safety": "medium",
            "required_creativity": "medium",
            "model": "medium"
        },
        "comment": "intermediate coding concept"
    },
    {
        "input": {"messages": [
            {"role":"user","content":"Design a machine learning pipeline to forecast stock prices using historical data, considering feature selection and model evaluation. What are the potential pitfalls?"}
        ]},
        "expected": {
            "thoughts": "Involves comprehensive understanding of machine learning, data preprocessing, and evaluation metrics.",
            "required_reasoning": "high",
            "required_truthfulness": "high",
            "required_safety": "high",
            "required_creativity": "high",
            "model": "large"
        },
        "comment": "advanced machine learning pipeline"
    }
]

# Lets build a routing that uses this task completion

In [137]:
def route_model(messages: list[dict], trace_id: str = None):
    """
    messages: [{"role":"user"|"assistant","content":"..."}]
    """
    resp = opper.functions.call(
        function_id=fn.id,
        input=ChatInput(messages=messages).model_dump(),
        parent_span_id=trace_id if trace_id else None,
    )
    out = resp.json_payload  # SDKs vary
    return out

# Example:
out = route_model(test_samples[0]["input"]["messages"])
print(out)

{'model': 'small', 'thoughts': 'User simply greeted; no complex or sensitive content requiring higher safety or reasoning.', 'required_reasoning': 'low', 'required_truthfulness': 'low', 'required_safety': 'low', 'required_creativity': 'low'}


# Lets test it

In [179]:
for sample in test_samples:
    resp = opper.functions.call(
        function_id=fn.id,
        input=ChatInput(messages=sample["input"]["messages"]).model_dump(),
    )
    out = resp.json_payload  
    print("Input:", sample["input"]["messages"])
    print("Output:", out["model"])
    print("Expected:", sample["expected"]["model"])
    print("Comparison:", "Match" if out["model"] == sample["expected"]["model"] else "Mismatch")
    print("--------------------------------")

Input: [{'role': 'user', 'content': 'Hello!'}]
Output: small
Expected: small
Comparison: Match
--------------------------------
Input: [{'role': 'user', 'content': 'What are the capitals of the Nordic countries, and how do their populations compare?'}]
Output: small
Expected: medium
Comparison: Mismatch
--------------------------------
Input: [{'role': 'user', 'content': 'Explain the differences between a list, a tuple, and a set in Python, and provide examples of when to use each.'}]
Output: medium
Expected: medium
Comparison: Match
--------------------------------
Input: [{'role': 'user', 'content': 'Design a machine learning pipeline to forecast stock prices using historical data, considering feature selection and model evaluation. What are the potential pitfalls?'}]
Output: medium
Expected: large
Comparison: Mismatch
--------------------------------


# Train

In [ ]:
# We populate the dataset of the function with these examples
for example in examples:

    try:
        creation = opper.datasets.create_entry(
            dataset_id=fn.dataset_id,  # Assuming fn has a dataset_id attribute
            input=example["input"],
            output=example["expected"],  # Using "expected" as the output
            comment=example["comment"]
        )
    except Exception as e:
        print(f"Error adding example: {e}")



name='model_router_chat' instructions='Choose the smallest model that will reliably answer the last user request, ' id='6ce4eea9-285d-49e4-aea8-7ff368133337' description=Unset() input_schema={'type': 'object', '$defs': {'ChatMessage': {'type': 'object', 'title': 'ChatMessage', 'required': ['role', 'content'], 'properties': {'role': {'enum': ['user', 'assistant'], 'type': 'string', 'title': 'Role'}, 'content': {'type': 'string', 'title': 'Content'}}}}, 'title': 'ChatInput', 'required': ['messages'], 'properties': {'messages': {'type': 'array', 'items': {'$ref': '#/$defs/ChatMessage'}, 'title': 'Messages'}}} output_schema={'type': 'object', 'title': 'RouterOutput', 'required': ['thoughts', 'required_reasoning', 'required_truthfulness', 'required_safety', 'required_creativity', 'model'], 'properties': {'model': {'enum': ['small', 'medium', 'large'], 'type': 'string', 'title': 'Model', 'description': "The selected model size: 'small' for trivial tasks, 'medium' for moderate tasks, 'large' 

# What if we could build a feedback loop that populates good samples?

In [180]:
# Model Map

model_map = {
    "small": "groq/gpt-oss-20b",
    "medium": "gcp/gemini-2.5-flash",
    "large": "anthropic/claude-opus-4.1"
}

# Create a trace

trace = opper.spans.create(
    name="conversation_test",
    meta={"models": str(model_map)},
)

# Conversation to test

new_input = {
    "messages": [
        {"role": "user", "content": "Hello! "},
        {"role": "assistant", "content": "Hello there what can I help you with today?"},
        {"role": "user", "content": "Work from the ground up to build a scientific model for how many people will inhabit Europe in 2050. Current numbers is that net growth is -5% per year. Give the final answer."},
    ]
}

# Select model to respond
model_selection = route_model(new_input["messages"], trace.id)
model_size = model_selection["model"]

print("Answer using model: ", model_size)

# Respond with the selected model
response = opper.stream(
    name="response",
    instructions="Respond with the last message in the conversation.",
    input=new_input["messages"],
    model=model_map[model_size],
    parent_span_id=trace.id
)

# Print the response
for event in response.result:

    # Each event is a FunctionStreamCallStreamPostResponseBody with 'data' containing the streaming chunk
    if hasattr(event, "data") and hasattr(event.data, "delta") and event.data.delta:
        print(event.data.delta, end="", flush=True)


# Get feedback from user
# Alternatively: This could be an async LLM as a judge OR an async expert classification

user_classification = input("Please classify the response as 'good' or 'bad': ")

# Add a new example to router task only if the signal is thumbs up

if user_classification == "good":

     # Log the thumbs up

    opper.span_metrics.create_metric(
            span_id=trace.id,  # Assuming entry_id can be used as span_id
            dimension="user_approved",
            value=1,  # 1 for thumbs up
            comment="User confirmed the example with a thumbs up."
        )
    print("Metric added for the completion.")

    # Add the example to the dataset

    new_example = {
        "input": new_input,
        "expected": model_selection,
        "comment": "User confirmed the example with a thumbs up."
    }

    try:
        entry_id = opper.datasets.create_entry(
            dataset_id=fn.dataset_id,  # Assuming fn has a dataset_id attribute
            input=new_example["input"],
            output=new_example["expected"],  # Using "expected" as the output
            comment=new_example["comment"]
        )
        print("New example added for manual confirmation with thumbs up.")

    except Exception as e:
        print(f"Error adding new example or metric: {e}")
else:
    print("User feedback was not thumbs up, example not added.")






Answer using model:  large
I'll build a scientific model to project Europe's population in 2050, working from current data and assumptions.

## Current Baseline (2024)
- Current European population: approximately 745 million
- Given: Net growth rate of -5% per year

## Model Construction

### Step 1: Exponential Decay Model
For population projection with a constant growth rate:
P(t) = P₀ × (1 + r)^t

Where:
- P(t) = Population at time t
- P₀ = Initial population (745 million)
- r = Growth rate (-0.05 or -5%)
- t = Time in years

### Step 2: Time Period
- Current year: 2024
- Target year: 2050
- Time period: 26 years

### Step 3: Calculation
P(2050) = 745,000,000 × (1 - 0.05)^26
P(2050) = 745,000,000 × (0.95)^26
P(2050) = 745,000,000 × 0.2653
P(2050) = 197,648,500

## Model Limitations & Considerations

**Note:** A -5% annual decline is extremely severe and unrealistic for the following reasons:
- Current European growth rate is actually around -0.1% to -0.2% annually
- Even during majo

In [181]:

from pydantic import BaseModel, Field

# Define the input schema for the conversation
class ConversationInput(BaseModel):
    conversation: str = Field(description="A fictive conversation between two people")

# Define the output schema for the sentiment analysis
class SentimentOutput(BaseModel):
    thoughts: str = Field(description="The thoughts of the model while determining the sentiment")
    sentiment: str = Field(description="The sentiment of the user response, e.g., 'positive', 'negative', 'neutral'")

# Fictive conversation
conversation_text = """
User: I really enjoyed the service today, everything was perfect!
Assistant: I'm glad to hear that! Is there anything else I can help you with?
User: No, that's all for now. Thank you!
"""

# Perform sentiment analysis on the user response
result = opper.call(
    name="analyzeSentiment",
    instructions="Determine the sentiment of the user response in the conversation",
    input_schema=ConversationInput,
    output_schema=SentimentOutput,
    input=ConversationInput(conversation=conversation_text)
)

print(result.json_payload)



{'thoughts': "The user expressed satisfaction with the service, using positive language such as 'really enjoyed' and 'everything was perfect.' Additionally, they closed the conversation politely with gratitude, which reinforces the positive sentiment.", 'sentiment': 'positive'}
